## Notebook summary

| Item | Details |
| --- | --- |
| Purpose | 01 - Original-Image Base Training |
| Model / workflow | DenseNet-121 |
| Input | 384x384 published crops (kneeKL224 source) |
| Loss | Cross-Entropy (CE) |
| Training / pipeline | Base training, full network |
| Result | (filled in after the run) |

# 01 - Original-Image Base Training (DenseNet-121)

Trains the base CE checkpoint on the **published** (pre-cropped) images. Notebook 02 adapts this
checkpoint to production YOLO ROIs, so run this one first.

- input: published crops, resized to 384x384
- loss: cross-entropy only
- sampler: inverse-frequency `WeightedRandomSampler`, so Grade 4 is not drowned out
- selection: validation `0.55*QWK + 0.30*macroF1 + 0.15*macroAP`
- the **test** split is never touched here

On completion the run directory is written to `SELECTED_CHECKPOINT.txt`, which Notebook 02 reads.
That pointer is what stops a stale run timestamp being hard-coded into the next stage.

In [1]:
!pip -q install "timm>=1.0" "h5py>=3.9"

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import json
import random
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, cohen_kappa_score, precision_recall_fscore_support
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from tqdm.auto import tqdm

Mounted at /content/drive


## Configuration

Everything tunable lives in this one cell. `PUBLISHED_ROOT` must contain `train/` and `val/`
subfolders, each holding `0/` .. `4/` grade folders of PNGs.

In [3]:
# ---- reproducibility -------------------------------------------------------
SEED = 42

# ---- data ------------------------------------------------------------------
INPUT_SIZE = 384
ROTATION_DEGREES = 5

# ---- optimisation ----------------------------------------------------------
EPOCHS = 12
BATCH_SIZE = 48
NUM_WORKERS = 2
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-3
PRETRAINED = True          # ImageNet initialisation for the base run

# ---- paths -----------------------------------------------------------------
PUBLISHED_ROOT = Path(
    "/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/"
    "extracted/KneeXrayData/ClsKLData/kneeKL224"
)
MODEL_ROOT = Path("/content/drive/MyDrive/Models/densenet121_original")

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H-%M-%S_%f_UTC")
RUN_DIR = MODEL_ROOT / RUN_TIMESTAMP

if not PUBLISHED_ROOT.exists():
    raise FileNotFoundError(PUBLISHED_ROOT)
RUN_DIR.mkdir(parents=True, exist_ok=False)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:   ", DEVICE)
print("Run dir:  ", RUN_DIR)

Device:    cuda
Run dir:   /content/drive/MyDrive/Models/densenet121_original/2026-08-23_06-24-04_305985_UTC


## Index the published images

One row per image. Only `train` and `val` are indexed — the test split is reserved for Notebook 03.

In [4]:
rows = []
for split in ("train", "val"):
    for grade in range(5):
        for path in sorted((PUBLISHED_ROOT / split / str(grade)).glob("*.png")):
            rows.append({"split": split, "grade": grade, "image_path": str(path)})

frame = pd.DataFrame(rows)
if frame.empty:
    raise RuntimeError(f"No PNGs found under {PUBLISHED_ROOT}")
print(frame.groupby(["split", "grade"]).size().unstack(fill_value=0))

grade     0     1     2    3    4
split                            
train  2286  1046  1516  757  173
val     328   153   212  106   27


## Preprocessing, dataset, and model

The validation transform is byte-identical to the one the API applies at inference time. Keeping
them in sync is what makes the reported metrics representative of the deployed service.

In [5]:
class OpenCVCLAHE:
    """LAB-space CLAHE. Identical to app/services/preprocessing_service.py."""

    def __call__(self, image_rgb):
        lab = cv2.cvtColor(np.asarray(image_rgb), cv2.COLOR_RGB2LAB)
        lightness, a, b = cv2.split(lab)
        lightness = cv2.createCLAHE(clipLimit=1.25, tileGridSize=(8, 8)).apply(lightness)
        return cv2.cvtColor(cv2.merge((lightness, a, b)), cv2.COLOR_LAB2RGB)


class SquarePad:
    """Pad to square with black borders, preserving aspect ratio."""

    def __call__(self, image_rgb):
        image = np.asarray(image_rgb)
        height, width = image.shape[:2]
        side = max(height, width)
        top, left = (side - height) // 2, (side - width) // 2
        return cv2.copyMakeBorder(
            image, top, side - height - top, left, side - width - left,
            cv2.BORDER_CONSTANT, value=(0, 0, 0),
        )


train_transform = transforms.Compose([
    OpenCVCLAHE(), SquarePad(), transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.50),
    transforms.RandomRotation(ROTATION_DEGREES),
    transforms.ColorJitter(brightness=0.08, contrast=0.08),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.10, scale=(0.02, 0.05), ratio=(0.5, 2.0), value=0),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    OpenCVCLAHE(), SquarePad(), transforms.ToPILImage(),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class ImageDataset(Dataset):
    def __init__(self, data, transform):
        self.data = data.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        image = cv2.imread(row.image_path, cv2.IMREAD_COLOR)
        if image is None:
            raise IOError(f"Cannot read image: {row.image_path}")
        return self.transform(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)), int(row.grade)


class DenseNet121Model(nn.Module):
    """Matches app/ml/models/densenet121_model.py so checkpoints load unchanged."""

    ARCHITECTURE = "timm_densenet121_linear_gradcam"

    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            "densenet121", pretrained=PRETRAINED, num_classes=5, drop_rate=0.20
        )

    @property
    def gradcam_target_layer(self):
        return self.backbone.features.norm5

    def forward(self, images):
        return self.backbone(images)


build_model = DenseNet121Model

## Train

One pass per epoch, then validate and keep the best checkpoint by selection score.

In [6]:
def selection_score(qwk, macro_f1, macro_ap):
    """Single scalar used to pick a checkpoint. Same weighting as the reference runs."""
    return 0.55 * qwk + 0.30 * macro_f1 + 0.15 * macro_ap


def score_predictions(labels, probabilities):
    labels = np.asarray(labels)
    probabilities = np.asarray(probabilities)
    predictions = probabilities.argmax(axis=1)
    _, _, macro_f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0
    )
    qwk = cohen_kappa_score(labels, predictions, weights="quadratic")
    macro_ap = average_precision_score(np.eye(5)[labels], probabilities, average="macro")
    return {
        "qwk": float(qwk),
        "macro_f1": float(macro_f1),
        "macro_ap": float(macro_ap),
        "selection": float(selection_score(qwk, macro_f1, macro_ap)),
    }


train_frame = frame[frame.split == "train"].copy()
val_frame = frame[frame.split == "val"].copy()

counts = np.bincount(train_frame.grade.to_numpy(), minlength=5)
weights = (1.0 / counts)[train_frame.grade.to_numpy()]
sampler = WeightedRandomSampler(
    torch.as_tensor(weights, dtype=torch.double), len(weights), replacement=True
)
train_loader = DataLoader(
    ImageDataset(train_frame, train_transform), batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True,
)
val_loader = DataLoader(
    ImageDataset(val_frame, val_transform), batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

model = build_model().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")


def validate():
    model.eval()
    labels, probabilities = [], []
    with torch.inference_mode():
        for images, batch_labels in val_loader:
            probs = F.softmax(model(images.to(DEVICE, non_blocking=True)).float(), dim=1)
            labels.extend(batch_labels.numpy())
            probabilities.extend(probs.cpu().numpy())
    return score_predictions(labels, probabilities)


best_score = -float("inf")
history = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    loss_sum, samples = 0.0, 0
    for images, labels in tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}"):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            loss = F.cross_entropy(model(images), labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item() * len(labels)
        samples += len(labels)
    scheduler.step()

    metrics = validate()
    row = {"epoch": epoch, "train_loss": loss_sum / samples, **metrics}
    history.append(row)
    print(json.dumps(row, indent=2))

    if metrics["selection"] > best_score:
        best_score = metrics["selection"]
        torch.save({
            "model_state_dict": model.state_dict(),
            "architecture": build_model.ARCHITECTURE,
            "loss_type": "ce",
            "epoch": epoch,
            "input_size": INPUT_SIZE,
            "selection": best_score,
        }, RUN_DIR / "best_model.pth")
        print(f"  => new best, selection={best_score:.4f}")

pd.DataFrame(history).to_csv(RUN_DIR / "history.csv", index=False)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


model.safetensors: reconstructing file:   0%|          |  0.00B / 32.3MB            

model.safetensors: downloading bytes:           |  0.00B            

epoch 1/12:   0%|          | 0/121 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Publish the checkpoint pointer

Notebook 02 reads `SELECTED_CHECKPOINT.txt` instead of a hard-coded timestamp, so re-running this
notebook automatically feeds the newer checkpoint forward.

In [ ]:
checkpoint_path = RUN_DIR / "best_model.pth"
(RUN_DIR / "SELECTED_CHECKPOINT.txt").write_text(str(checkpoint_path))
(RUN_DIR / "run_config.json").write_text(json.dumps({
    "stage": "01_original",
    "input_size": INPUT_SIZE,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "loss": "cross_entropy",
    "best_selection": best_score,
}, indent=2))

print("Best checkpoint:", checkpoint_path)
print("Pointer written:", RUN_DIR / "SELECTED_CHECKPOINT.txt")
print("Next: run 02_train_paired_roi.ipynb")